# ShExMap Validation & Materialisation

This notebook demonstrates the full ShExMap workflow:
1. **Query** the QLever triplestore for a pairing — retrieves both ShEx schemas, sample Turtle data, and focus nodes.
2. **Validate** the source RDF against the source ShEx and extract Map bindings.
3. **Materialise** the target RDF by applying the bindings to the target ShEx.
4. **Pretty-print** the resulting Turtle.

Requires the ShExMap Repository stack to be running (`docker compose up`).

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
SPARQL_ENDPOINT = "http://localhost:7001/sparql"   # QLever direct
API_BASE        = "http://localhost:8080/api/v1"   # via nginx (host-mapped port)

import json, textwrap, requests
from pprint import pprint

## 1 — Query the triplestore for a shexmap, sample data, and focus iri

In [ ]:
def getShexcFromAPI(title):
    SPARQL_QUERY = """
PREFIX dct:      <http://purl.org/dc/terms/>
PREFIX shexmap:  <https://w3id.org/shexmap/ontology#>
PREFIX schema:   <https://schema.org/>

SELECT ?title (?content as ?shexMap) (?ttl as ?sampleTurtleData)
{
    ?map a shexmap:ShExMap ;
        dct:title ?title;
        shexmap:currentVersion ?shexc;
        shexmap:sampleTurtleData ?ttl.
    ?shexc shexmap:versionContent ?content .
    FILTER (CONTAINS(STR(?title), \""""+title+"""\"))
}
    """

    resp = requests.get(
        SPARQL_ENDPOINT,
        params={"query": SPARQL_QUERY, "format": "json"},
        timeout=30,
    )
    resp.raise_for_status()
    results = resp.json()["results"]["bindings"]
    # create an object from the results with the title, shexc, and ttl
    result = None
    if results:
        result = {
            "title": results[0]["title"]["value"],
            "shexMap": results[0]["shexMap"]["value"],
            "sampleTurtleData": results[0]["sampleTurtleData"]["value"]
        }

    return result


In [56]:
#let's get the shexc content for the DAM BP and FHIR BP maps
dam_bp = getShexcFromAPI("DAM BP")
dam_bp['focusIRI'] = '<tag:b0>'

fhir_bp = getShexcFromAPI("FHIR BP")
fhir_bp['focusIRI'] = '<tag:BPfhir123>'


### Load the SULO Mapping

In [65]:
# load the shexc content as JSON from the sparql/files/sulo-bp.shex file and print it
sulo_bp = {}
with open("./files/sulo-bp.shex", "r") as f:
    sulo_bp['shexMap'] = f.read()

with open("./files/sulo-bp.ttl", "r") as f:
    sulo_bp['sampleTurtleData'] = f.read()

sulo_bp['focusIRI'] = '<tag:b1>'


## 3 — Validate and materialise the first target RDF

We `POST` to `/api/v1/validate` with:
- `sourceShEx` + `sourceRdf` + `sourceNode` → validates & extracts Map variable bindings
- `targetShEx` + `targetNode` → materialises new RDF using those bindings

In [57]:
def MateralizeValidation(src_shex, src_turtle, src_focus, tgt_shex, tgt_focus): 
    payload = {
        "sourceShEx":  src_shex,
        "sourceRdf":   src_turtle,
        "sourceNode":  src_focus,
        "targetShEx":  tgt_shex,
        "targetNode":  tgt_focus,
    }

    resp = requests.post(
        f"{API_BASE}/validate",
        json=payload,
        timeout=30,
    )
    resp.raise_for_status()
    result = resp.json()
    return result

In [ ]:
# Let's materialize the SULO map from the DAM BP map and sample data. 
result = MateralizeValidation(
    src_shex=dam_bp['shexMap'],
    src_turtle=dam_bp['sampleTurtleData'], 
    src_focus=dam_bp['focusIRI'],
    tgt_shex=sulo_bp['shexMap'],
    tgt_focus=sulo_bp['focusIRI']
)
# check that the result is valid and print the errors if not
if not result['valid']:
    print("Validation failed with errors:")
    pprint(result['errors'])
else :
    print("Validation succeeded!")

# here we get the target RDF from the result 
sulo_target_rdf = result.get("targetRdf", "")
print(sulo_target_rdf)


Validation succeeded!
@prefix : <http://w3id.org/sulo/ext/>.
@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.
@prefix bp: <http://shex.io/extensions/Map/#BPDAM->.
@prefix Map: <http://shex.io/extensions/Map/#>.
@prefix sulo: <https://w3id.org/sulo/>.
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>.

<tag:b1> sulo:hasFeature _:b0.
_:b0 sulo:hasValue "Alice".
<tag:b1> sulo:hasFeature _:b1.
_:b1 sulo:hasValue "Walker".
<tag:b1> sulo:hasFeature _:b2.
_:b2 sulo:hasValue "110"^^xsd:float;
    sulo:hasPart _:b3.
_:b3 rdfs:label "mmHg".
<tag:b1> sulo:hasFeature _:b4.
_:b4 sulo:hasValue "70"^^xsd:float;
    sulo:hasPart _:b5.
_:b5 rdfs:label "mmHg".



## 4 — Validate and materialise the second target RDF


In [ ]:
# Validation chain
result = MateralizeValidation(
    src_shex=sulo_bp['shexMap'],
    src_turtle=sulo_target_rdf,  ## we use the output of the previous validation as input for this one
    src_focus=sulo_bp['focusIRI'],
    tgt_shex=fhir_bp['shexMap'],
    tgt_focus=fhir_bp['focusIRI']
)

# check that the result is valid and print the errors if not
if not result['valid']:
    print("Validation failed with errors:")
    pprint(result['errors'])
else :
    print("Validation succeeded!")

fhir_target_rdf = result.get("targetRdf", "")
print(fhir_target_rdf)

Validation succeeded!
@prefix fhir: <http://hl7.org/fhir-rdf/>.
@prefix sct: <http://snomed.info/sct/>.
@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.
@prefix bp: <http://shex.io/extensions/Map/#BPDAM->.
@prefix Map: <http://shex.io/extensions/Map/#>.

<tag:BPfhir123> a fhir:Observation;
    fhir:subject _:b0.
_:b0 fhir:givenName "Alice";
    fhir:familyName "Walker".
<tag:BPfhir123> fhir:coding _:b1.
_:b1 fhir:code sct:Blood_Pressure.
<tag:BPfhir123> fhir:component _:b2.
_:b2 a fhir:Observation;
    fhir:coding _:b3.
_:b3 fhir:code sct:Systolic_Blood_Pressure.
_:b2 fhir:valueQuantity _:b4.
_:b4 a fhir:Quantity;
    fhir:value "110"^^xsd:float;
    fhir:units "mmHg".
<tag:BPfhir123> fhir:component _:b5.
_:b5 a fhir:Observation;
    fhir:coding _:b6.
_:b6 fhir:code sct:Diastolic_Blood_Pressure.
_:b5 fhir:valueQuantity _:b7.
_:b7 a fhir:Quantity;
    fhir:value "70"^^xsd:float;
    fhir:units "mmHg".



## 5 — Validate and materialise the third target RDF (roundtrip)


In [ ]:
# Validation chain
result = MateralizeValidation(
    src_shex=fhir_bp['shexMap'],
    src_turtle=fhir_target_rdf,  ## we use the output of the previous validation as input for this one
    src_focus=fhir_bp['focusIRI'],
    tgt_shex=dam_bp['shexMap'],
    tgt_focus=dam_bp['focusIRI']
)

# check that the result is valid and print the errors if not
if not result['valid']:
    print("Validation failed with errors:")
    pprint(result['errors'])
else :
    print("Validation succeeded!")

dam_target_rdf = result.get("targetRdf", "")
print(dam_target_rdf)

Validation succeeded!
@prefix : <http://dam.example/med#>.
@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.
@prefix bp: <http://shex.io/extensions/Map/#BPDAM->.
@prefix Map: <http://shex.io/extensions/Map/#>.

<tag:b0> :name "Walker, Alice";
    :systolic _:b0.
_:b0 :value "110"^^xsd:float;
    :units "mmHg".
<tag:b0> :diastolic _:b1.
_:b1 :value "70"^^xsd:float;
    :units "mmHg".

